# Tune Omni 600M for Omni Mail

Start with the [lab guide](README.md) for setup and labeling. This notebook
uses the same local classifier as the app. Run all the cells in order with the
fictional CSV first. The Codex writer is optional and starts disabled.

## 1. Load the model and run one decision

The app's DSPy signature names three inputs: sender, subject, and body.
Its output, `needs_action`, is a `Noul` decision with a Yes score and a
Boolean value. The profile supplies the instructions, the question, and the
cutoff. Here is one fictional email, separate from the labeled dataset.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys
from tempfile import mkdtemp
from time import perf_counter

# For a private notebook copy, set this to your Omni Mail checkout.
project = Path.cwd().resolve()
if project.name == "omni-mail-lab":
    project = project.parent
if not (project / "omni_mail").is_dir():
    raise RuntimeError("Set project to the Omni Mail checkout.")
sys.path.insert(0, str(project))

import dspy
from dspy.experimental import ReAnchor
from omni_mail.classifier import INPUTS, LocalClassifier, decision_program
from omni_mail.lab import (
    WEIGHTSPACE_TENSORS, action_credit, examples_for, load_rows,
    predict, score, split_rows, tune_weights,
)
from omni_mail.profiles import STOCK, load_profile, save_profile

profile = STOCK["omni-600m"]
start = perf_counter()
model = LocalClassifier(profile)
print(f"Loaded {profile.label} in {perf_counter() - start:.2f} seconds.")
print("DSPy fields:", ", ".join(model.program.signature.fields))
print("Instructions:", profile.instructions)
print("Question:", profile.question)
example = dict(
    sender="planning@example.invalid",
    subject="Please confirm the delivery window",
    body="Can you confirm by Wednesday whether your team can receive the delivery on Friday?",
)
print("Input:", example)
decision = model.predict(**example)
print("Expected label for this example: Action")
print("Yes score:", decision.probability, "| Cutoff:", profile.cutoff,
      "| Label:", "Action" if decision.value else "No Action")

## 2. Load reviewed labels and split the data

Set `mail_csv` to your private CSV when you're ready to use your own mail.
Follow the [batch labeling workflow](README.md#label-in-batches) and use the
same `group_id` for related messages and repeated templates. The split
allocates roughly 60%, 20%, and 20% of **groups**, so the row counts can vary.

Use training to inspect errors and fit changes, validation to choose which
changes to keep, and the final split once the profile is frozen.

In [ ]:
sample_csv = project / "omni-mail-lab/examples/sample-labels.csv"
mail_csv = sample_csv  # Or set Path("/path/to/private/label-batch.csv")
private_dir = Path.home() / "Library/Application Support/Omni Mail Lab"
private_dir.mkdir(parents=True, exist_ok=True, mode=0o700)
run_dir = Path(mkdtemp(prefix=f"{profile.key}-", dir=private_dir))

train_rows, validation_rows, final_rows = split_rows(load_rows(mail_csv))
train = examples_for(model, train_rows)
validation = examples_for(model, validation_rows)
for name, rows in (("Training", train_rows), ("Validation", validation_rows)):
    positives = sum(row["label"] == "yes" for row in rows)
    print(f"{name}: {len(rows)} rows ({positives} Action, {len(rows) - positives} No Action)")
print(f"Final: {len(final_rows)} rows reserved")

## 3. Time the baseline and review training mistakes

The original model and its original cutoff establish the baseline. The score
charges two points for each missed action and one for each false alert. A
lower penalty is better. The notebook also tracks how many actions you
would miss so a big drop in false alerts cannot hide extra misses.

Inspect the training mistakes before tuning. For your own CSV, start with
IDs and look them up locally. Enable `show_training_text` in a private copy
if you want the notebook to display the original email text.

In [ ]:
start = perf_counter()
train_predictions = predict(model.program, train)
print(f"Training inference: {perf_counter() - start:.2f} seconds for {len(train)} emails")
print("Training baseline:", score(train, train_predictions))

start = perf_counter()
original_validation = score(validation, predict(model.program, validation))
print(f"Validation inference: {perf_counter() - start:.2f} seconds")
print("Validation baseline:", original_validation)

show_training_text = mail_csv.resolve() == sample_csv
errors = [(row, yes) for row, yes in zip(train_rows, train_predictions, strict=True)
          if (row["label"] == "yes") != yes]
print("Training errors:", len(errors))
for row, predicted_yes in errors[:5]:
    print(row["id"], "| Expected:", row["label"], "| Predicted:", "yes" if predicted_yes else "no")
    if show_training_text:
        print({field: row[field] for field in INPUTS})

## 4. Use ReAnchor to fit the cutoff

ReAnchor uses the training decisions to find a cutoff. The scoring metric
gives a correctly identified Action twice the credit of a correctly
identified No Action. The model weights stay fixed. Use validation to keep
the fitted cutoff only if it lowers the penalty without increasing missed
actions.

In [ ]:
cutoff_profile = profile
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=True)
try:
    calibration = decision_program(model.backend, profile, cache=True)
    fitted = ReAnchor(metric=action_credit, num_threads=1).compile(calibration, trainset=train)
    new_cutoff = fitted.fields["needs_action"]["threshold"]
    anchored = score(validation, predict(fitted, validation))
finally:
    dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=False)

print("Fitted cutoff:", new_cutoff, "| Validation:", anchored)
if anchored["penalty"] < original_validation["penalty"] and anchored["missed"] <= original_validation["missed"]:
    cutoff_profile = replace(profile, cutoff=float(new_cutoff))
print("Cutoff kept for the next step:", cutoff_profile.cutoff)

## 5. Preview the optional WeightSpace run

WeightSpace asks a Codex writer to propose changes to a small set of model
weights. The search uses the training examples while the cutoff stays fixed.
The notebook then checks the proposed patch on validation and keeps it
only if the penalty falls without extra missed actions.

Run this preview before enabling Codex. It shows the pool of eligible
training examples, which tensors the writer may edit, and the search
budget. The writer can receive training email text, labels, model or tensor
information, and search feedback. Validation and final email text stay local.
See the [writer setup](README.md#4-enable-the-optional-weightspace-step).

In [ ]:
writer_model = ""  # Empty uses the signed-in Codex CLI default.
max_passes = 3

print("Writer:", writer_model or "Codex CLI default")
print("Maximum forward passes:", max_passes, "| Maximum writer turns:", max_passes * 3)
print("Fixed cutoff:", cutoff_profile.cutoff)
print("Editable tensors:", *WEIGHTSPACE_TENSORS[profile.key], sep="\n  ")
print("Eligible training IDs:", ", ".join(row["id"] for row in train_rows))
if show_training_text:
    print("Example eligible inputs:", train[0].inputs().toDict())
    print("Example eligible label:", train[0].label)

### Run WeightSpace

Set `use_codex = True` only after checking the preview and signing in to
Codex. Start with the fictional sample. When the run finishes you will see
validation scores for the cutoff-only model and the proposed patch.

In [ ]:
use_codex = False
selected = cutoff_profile
try:
    if use_codex:
        patched = tune_weights(
            model, decision_program(model.backend, cutoff_profile), train, validation,
            run_dir / "weightspace", writer_model, max_passes, cutoff_profile.cutoff,
        )
        if patched:
            selected = patched
            print("Accepted patch:", patched.patch)
        else:
            print("Kept the cutoff-only profile.")
    else:
        print("WeightSpace skipped. Continuing with the cutoff-only profile.")
finally:
    model.close()

## 6. Freeze the profile and compare on reserved email

Export the chosen profile before looking at the final scores. Compare
stock with the profile reloaded using the app's loader. If you accepted a
patch and changed the cutoff, also run the cutoff-only profile to see what
each change contributed. If the cutoff stayed at stock, the stock row
already shows the cutoff-only result.

Treat this comparison as the final report. Use a new reserved set if
you make further choices based on what you see here. The fictional fixture
checks that the steps run. Use your own labeled mail to measure usefulness.

In [ ]:
exported = run_dir / "profile.json"
save_profile(exported, selected)
loaded = load_profile(exported)

comparisons = [("Stock", profile)]
if loaded.patch and cutoff_profile.cutoff != profile.cutoff:
    comparisons.append(("Cutoff only", cutoff_profile))
comparisons.append(("Reloaded profile", loaded))

for name, comparison in comparisons:
    local_model = LocalClassifier(comparison)
    try:
        final = examples_for(local_model, final_rows)
        started = perf_counter()
        results = score(final, predict(local_model.program, final))
        print(f"{name}: {results} | {perf_counter() - started:.2f} seconds")
    finally:
        local_model.close()

print(f'Load in Omni Mail with --model {profile.key} --profile "{exported}"')